In [1]:
import json
from pathlib import Path

data_path = Path("../data/products.json")

with data_path.open("r", encoding="utf-8") as file:
    products = json.load(file)

print(f"Loaded {len(products)} products")
print(products[0])


Loaded 62 products
{'id': 'cl-001', 'name': 'Classic White Cotton T-Shirt', 'category': 't-shirt', 'gender': 'unisex', 'colors': ['white'], 'style': ['casual', 'basic'], 'material': ['cotton'], 'season': ['spring', 'summer'], 'price': 799, 'rating': 4.6, 'popularity': 98, 'sizes': ['S', 'M', 'L', 'XL'], 'stock': 42, 'imageUrl': 'https://images.unsplash.com/photo-1521572163474-6864f9cf17ab?auto=format&fit=crop&w=600&q=80'}


In [2]:
import pandas as pd

products_df = pd.DataFrame(products)

print("Rows and columns:", products_df.shape)
print("Available fields:")
print(products_df.columns.tolist())

products_df.head(3)


Rows and columns: (62, 14)
Available fields:
['id', 'name', 'category', 'gender', 'colors', 'style', 'material', 'season', 'price', 'rating', 'popularity', 'sizes', 'stock', 'imageUrl']


,id,name,category,gender,colors,style,material,season,price,rating,popularity,sizes,stock,imageUrl
0,cl-001,Classic White Cotton T-Shirt,t-shirt,unisex,[white],"[casual, basic]",[cotton],"[spring, summer]",799,4.6,98,"[S, M, L, XL]",42,https://images.unsplash.com/photo-152157216347...
1,cl-002,Oversized Black Graphic T-Shirt,t-shirt,unisex,[black],"[casual, streetwear, oversized]",[cotton],"[spring, summer, autumn]",999,4.5,94,"[S, M, L, XL]",31,https://images.unsplash.com/photo-150334150425...
2,cl-003,Sky Blue Linen Shirt,shirt,men,[blue],"[casual, smart-casual]",[linen],"[spring, summer]",1599,4.7,91,"[S, M, L, XL]",19,https://images.unsplash.com/photo-160281031838...


In [3]:
products_df.isnull().sum()



id            0
name          0
category      0
gender        0
colors        0
style         0
material      0
season        0
price         0
rating        0
popularity    0
sizes         0
stock         0
imageUrl      0
dtype: int64

In [4]:
feature_columns = [
    "category",
    "gender",
    "colors",
    "style",
    "material",
    "season"
]

def build_feature_text(row):
    parts = []

    for column in feature_columns:
        value = row[column]

        if isinstance(value, list):
            parts.extend(str(item).lower() for item in value)
        else:
            parts.append(str(value).lower())

    return " ".join(parts)

products_df["feature_text"] = products_df.apply(build_feature_text, axis=1)

products_df[["name", "feature_text"]].head()

,name,feature_text
0,Classic White Cotton T-Shirt,t-shirt unisex white casual basic cotton sprin...
1,Oversized Black Graphic T-Shirt,t-shirt unisex black casual streetwear oversiz...
2,Sky Blue Linen Shirt,shirt men blue casual smart-casual linen sprin...
3,Slim Fit Dark Blue Jeans,jeans unisex blue casual everyday denim cotton...
4,High-Waist Straight Blue Jeans,jeans women blue casual everyday high-waist de...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

feature_matrix = vectorizer.fit_transform(
    products_df["feature_text"]
)

print("Matrix shape:", feature_matrix.shape)

Matrix shape: (62, 109)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(feature_matrix)
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (62, 62)


In [7]:
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=products_df["name"],
    columns=products_df["name"]
)

similarity_df.round(2)

name,Classic White Cotton T-Shirt,Oversized Black Graphic T-Shirt,Sky Blue Linen Shirt,Slim Fit Dark Blue Jeans,High-Waist Straight Blue Jeans,Floral Summer Midi Dress,Elegant Black Evening Dress,Warm Beige Knit Sweater,Olive Lightweight Bomber Jacket,Navy Blue Zip Hoodie,...,Halter Neck Top,Pleated Palazzo Trousers,Oversized Graphic Sweatshirt,Tiered Maxi Skirt,Satin Co-Ord Set,Quilted Cropped Jacket,Ruched Bodycon Dress,Straight-Leg Mom Jeans,Oversized Zip Hoodie,Relaxed Canvas Workwear Jacket
name,,,,,,,,,,,,,,,,,,,,,
Classic White Cotton T-Shirt,1.00,0.56,0.36,0.33,0.14,0.21,0.00,0.18,0.06,0.45,...,0.07,0.13,0.04,0.20,0.06,0.05,0.07,0.15,0.19,0.20
Oversized Black Graphic T-Shirt,0.56,1.00,0.34,0.35,0.16,0.11,0.09,0.19,0.21,0.25,...,0.12,0.22,0.42,0.10,0.12,0.08,0.15,0.16,0.54,0.22
Sky Blue Linen Shirt,0.36,0.34,1.00,0.29,0.23,0.15,0.00,0.08,0.16,0.09,...,0.06,0.31,0.00,0.06,0.05,0.10,0.06,0.34,0.00,0.02
Slim Fit Dark Blue Jeans,0.33,0.35,0.29,1.00,0.71,0.10,0.07,0.20,0.12,0.26,...,0.06,0.13,0.10,0.09,0.06,0.12,0.08,0.62,0.21,0.22
High-Waist Straight Blue Jeans,0.14,0.16,0.23,0.71,1.00,0.11,0.10,0.08,0.10,0.11,...,0.08,0.14,0.12,0.10,0.08,0.13,0.10,0.54,0.08,0.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Quilted Cropped Jacket,0.05,0.08,0.10,0.12,0.13,0.08,0.20,0.10,0.43,0.11,...,0.11,0.19,0.12,0.04,0.11,1.00,0.13,0.14,0.18,0.33
Ruched Bodycon Dress,0.07,0.15,0.06,0.08,0.10,0.21,0.47,0.02,0.02,0.02,...,0.85,0.23,0.11,0.09,0.28,0.13,1.00,0.10,0.06,0.04
Straight-Leg Mom Jeans,0.15,0.16,0.34,0.62,0.54,0.11,0.10,0.08,0.10,0.12,...,0.08,0.15,0.12,0.11,0.08,0.14,0.10,1.00,0.08,0.09


In [8]:
def recommend_products(product_id, n=5):
    try:
        product_position = products_df["id"].tolist().index(product_id)
    except ValueError:
        raise ValueError(f"Product ID not found: {product_id}")

    scores = similarity_matrix[product_position]

    ranked_positions = scores.argsort()[::-1]

    ranked_positions = [
        position
        for position in ranked_positions
        if position != product_position
    ][:n]

    recommendations = products_df.iloc[ranked_positions][
        ["id", "name", "category", "colors", "style", "price"]
    ].copy()

    recommendations["similarity_score"] = (
        scores[ranked_positions].round(3)
    )

    return recommendations.reset_index(drop=True)

In [9]:
recommend_products("cl-002", n=5)

,id,name,category,colors,style,price,similarity_score
0,cl-013,Oversized Graphic T-Shirt,t-shirt,"[black, white]","[streetwear, oversized]",999,0.782
1,cl-001,Classic White Cotton T-Shirt,t-shirt,[white],"[casual, basic]",799,0.559
2,cl-061,Oversized Zip Hoodie,hoodie,"[grey, black, cream]","[streetwear, oversized]",1999,0.543
3,cl-011,Relaxed Cotton Cargo Pants,trousers,[khaki],"[casual, utility, streetwear]",1999,0.476
4,cl-021,Heavyweight Pullover Hoodie,hoodie,"[grey, black]","[streetwear, oversized]",2199,0.442


In [10]:
products_df["search_text"] = (
    products_df["name"].fillna("")
    + " "
    + products_df["feature_text"]
)

search_vectorizer = TfidfVectorizer()

search_matrix = search_vectorizer.fit_transform(
    products_df["search_text"]
)

In [11]:
def normalize_search_query(query):
    query = query.lower().strip()

    synonyms = {
        "tshirt": "t-shirt",
        "t shirt": "t-shirt",
        "tee": "t-shirt",
        "tees": "t-shirt",
        "pants": "trousers",
        "pant": "trousers",
        "trouser": "trousers",
        "jean": "jeans",
        "hoodies": "hoodie",
        "shirts": "shirt",
        "jackets": "jacket"
    }

    for original, standard in sorted(
        synonyms.items(),
        key=lambda item: len(item[0]),
        reverse=True
    ):
        query = query.replace(original, standard)

    return query

In [12]:
def search_products(query, n=10):
    normalized_query = normalize_search_query(query)

    query_vector = search_vectorizer.transform(
        [normalized_query]
    )

    if query_vector.nnz == 0:
        return pd.DataFrame()

    scores = cosine_similarity(
        query_vector,
        search_matrix
    ).ravel()

    results = products_df.copy()
    results["search_score"] = scores

    results = results[results["stock"] > 0]
    results = results[results["search_score"] > 0]

    results = results.sort_values(
        ["search_score", "popularity"],
        ascending=[False, False]
    )

    return results[
        [
            "id",
            "name",
            "category",
            "colors",
            "style",
            "price",
            "stock",
            "search_score"
        ]
    ].head(n).reset_index(drop=True)

In [13]:
search_products("white t-shirt")

,id,name,category,colors,style,price,stock,search_score
0,cl-001,Classic White Cotton T-Shirt,t-shirt,[white],"[casual, basic]",799,42,0.719620
1,cl-013,Oversized Graphic T-Shirt,t-shirt,"[black, white]","[streetwear, oversized]",999,58,0.538318
2,cl-032,Striped Rugby Shirt,shirt,"[green, white]","[college, casual]",1599,41,0.529139
3,cl-028,Relaxed Oxford Shirt,shirt,"[light-blue, white]","[classic, smart-casual]",1499,44,0.517325
4,cl-014,Relaxed Linen Shirt,shirt,"[beige, white]","[casual, resort]",1599,42,0.503846
5,cl-002,Oversized Black Graphic T-Shirt,t-shirt,[black],"[casual, streetwear, oversized]",999,31,0.370890
6,cl-035,Premium Henley Shirt,shirt,"[maroon, grey]","[casual, basic]",1099,47,0.369324
7,cl-003,Sky Blue Linen Shirt,shirt,[blue],"[casual, smart-casual]",1599,19,0.338700
8,cl-020,Camp Collar Resort Shirt,shirt,"[green, cream]","[resort, relaxed]",1399,29,0.322696
9,cl-017,Textured Knit Polo Shirt,polo-shirt,"[brown, cream]","[smart-casual, minimal]",1799,31,0.293827


In [14]:
search_products("black pants")

,id,name,category,colors,style,price,stock,search_score
0,cl-054,Pleated Palazzo Trousers,trousers,"[beige, black]","[minimal, smart-casual]",1599,45,0.582530
1,cl-023,Pleated Relaxed Trousers,trousers,"[beige, charcoal]","[smart-casual, minimal]",1799,35,0.512862
2,cl-034,Linen Blend Drawstring Trousers,trousers,"[white, sand]","[resort, relaxed]",1699,28,0.422135
3,cl-015,Parachute Cargo Pants,trousers,"[olive, black]","[streetwear, utility]",1899,37,0.375547
4,cl-011,Relaxed Cotton Cargo Pants,trousers,[khaki],"[casual, utility, streetwear]",1999,18,0.288476
5,cl-002,Oversized Black Graphic T-Shirt,t-shirt,[black],"[casual, streetwear, oversized]",999,31,0.164958
6,cl-007,Elegant Black Evening Dress,dress,[black],"[formal, party, elegant]",3299,12,0.145120
7,cl-013,Oversized Graphic T-Shirt,t-shirt,"[black, white]","[streetwear, oversized]",999,58,0.087201
8,cl-018,Performance Jogger Pants,joggers,"[charcoal, black]","[athleisure, sporty]",1299,64,0.079356
9,cl-037,Ribbed Crop Top,top,"[black, white, beige]","[casual, minimal]",799,61,0.077445


In [16]:
search_products("jogger")

,id,name,category,colors,style,price,stock,search_score
0,cl-018,Performance Jogger Pants,joggers,"[charcoal, black]","[athleisure, sporty]",1299,64,0.382131


In [18]:
search_products("white  jogger")

,id,name,category,colors,style,price,stock,search_score
0,cl-018,Performance Jogger Pants,joggers,"[charcoal, black]","[athleisure, sporty]",1299,64,0.337262
1,cl-001,Classic White Cotton T-Shirt,t-shirt,[white],"[casual, basic]",799,42,0.221125
2,cl-013,Oversized Graphic T-Shirt,t-shirt,"[black, white]","[streetwear, oversized]",999,58,0.105168
3,cl-032,Striped Rugby Shirt,shirt,"[green, white]","[college, casual]",1599,41,0.103375
4,cl-051,Cropped Denim Shacket,jacket,"[blue, white]","[casual, streetwear]",2399,29,0.102604
5,cl-028,Relaxed Oxford Shirt,shirt,"[light-blue, white]","[classic, smart-casual]",1499,44,0.101067
6,cl-014,Relaxed Linen Shirt,shirt,"[beige, white]","[casual, resort]",1599,42,0.098434
7,cl-037,Ribbed Crop Top,top,"[black, white, beige]","[casual, minimal]",799,61,0.093403
8,cl-046,Pleated Tennis Skirt,skirt,"[white, navy]","[sporty, preppy]",1199,38,0.087564
9,cl-056,Tiered Maxi Skirt,skirt,"[yellow, white]","[boho, resort]",1999,24,0.086013


In [19]:
search_products("trouser")

,id,name,category,colors,style,price,stock,search_score
0,cl-023,Pleated Relaxed Trousers,trousers,"[beige, charcoal]","[smart-casual, minimal]",1799,35,0.588423
1,cl-054,Pleated Palazzo Trousers,trousers,"[beige, black]","[minimal, smart-casual]",1599,45,0.577071
2,cl-034,Linen Blend Drawstring Trousers,trousers,"[white, sand]","[resort, relaxed]",1699,28,0.484329
3,cl-011,Relaxed Cotton Cargo Pants,trousers,[khaki],"[casual, utility, streetwear]",1999,18,0.330978
4,cl-015,Parachute Cargo Pants,trousers,"[olive, black]","[streetwear, utility]",1899,37,0.327322


In [20]:
trouser_products = products_df[
    products_df["category"].str.lower() == "trousers"
]

print("Number of trousers:", len(trouser_products))
trouser_products[["id", "name", "price", "stock"]]


Number of trousers: 5


,id,name,price,stock
10,cl-011,Relaxed Cotton Cargo Pants,1999,18
14,cl-015,Parachute Cargo Pants,1899,37
22,cl-023,Pleated Relaxed Trousers,1799,35
33,cl-034,Linen Blend Drawstring Trousers,1699,28
53,cl-054,Pleated Palazzo Trousers,1599,45


In [21]:
trouser_products = products_df[
    products_df["category"].str.lower() == "trousers"
]

print("Number of trousers:", len(trouser_products))
trouser_products[["id", "name", "price", "stock"]]

Number of trousers: 5


,id,name,price,stock
10,cl-011,Relaxed Cotton Cargo Pants,1999,18
14,cl-015,Parachute Cargo Pants,1899,37
22,cl-023,Pleated Relaxed Trousers,1799,35
33,cl-034,Linen Blend Drawstring Trousers,1699,28
53,cl-054,Pleated Palazzo Trousers,1599,45


In [22]:
trouser_products = products_df[
    products_df["category"].str.lower() == "trousers"
]

print("Number of trousers:", len(trouser_products))

trouser_products[["id", "name", "category"]]

Number of trousers: 5


,id,name,category
10,cl-011,Relaxed Cotton Cargo Pants,trousers
14,cl-015,Parachute Cargo Pants,trousers
22,cl-023,Pleated Relaxed Trousers,trousers
33,cl-034,Linen Blend Drawstring Trousers,trousers
53,cl-054,Pleated Palazzo Trousers,trousers


In [23]:
trouser_products = products_df[
    products_df["category"].str.lower() == "trousers"
]

print("Number of trousers:", len(trouser_products))
trouser_products[["id", "name", "price", "stock"]]

Number of trousers: 5


,id,name,price,stock
10,cl-011,Relaxed Cotton Cargo Pants,1999,18
14,cl-015,Parachute Cargo Pants,1899,37
22,cl-023,Pleated Relaxed Trousers,1799,35
33,cl-034,Linen Blend Drawstring Trousers,1699,28
53,cl-054,Pleated Palazzo Trousers,1599,45


In [24]:
trouser_products = products_df[
    products_df["category"].str.lower() == "trousers"
]

print("Number of trousers:", len(trouser_products))

trouser_products[["id", "name", "category"]]


Number of trousers: 5


,id,name,category
10,cl-011,Relaxed Cotton Cargo Pants,trousers
14,cl-015,Parachute Cargo Pants,trousers
22,cl-023,Pleated Relaxed Trousers,trousers
33,cl-034,Linen Blend Drawstring Trousers,trousers
53,cl-054,Pleated Palazzo Trousers,trousers


In [28]:
products_df["category"].value_counts()

category
shirt         6
dress         6
jacket        6
jeans         5
trousers      5
t-shirt       3
hoodie        3
skirt         3
sweater       2
vest          2
blazer        2
top           2
co-ord-set    2
lounge-set    1
polo-shirt    1
joggers       1
overshirt     1
chinos        1
tank-top      1
shorts        1
kurta         1
activewear    1
leggings      1
cardigan      1
blouse        1
waistcoat     1
tracksuit     1
sweatshirt    1
Name: count, dtype: int64

In [29]:
products_df["gender"].value_counts()

gender
women     28
men       26
unisex     8
Name: count, dtype: int64

In [30]:
products_df["colors"].explode().value_counts()

colors
black         25
white         15
cream          9
blue           7
grey           7
beige          6
olive          6
navy           5
brown          5
charcoal       4
pink           3
green          3
khaki          2
dark-blue      2
sand           2
light-blue     2
sky-blue       2
yellow         2
red            2
purple         1
rust           1
maroon         1
emerald        1
sage-green     1
lavender       1
champagne      1
Name: count, dtype: int64